In [5]:
!git clone -b disease_trends https://github.com/Husayn01/Team-Q.git

Cloning into 'Team-Q'...
remote: Enumerating objects: 133, done.
remote: Counting objects: 100% (133/133), done.
remote: Compressing objects: 100% (101/101), done.
remote: Total 133 (delta 26), reused 128 (delta 21), pack-reused 0 (from 0)
Receiving objects: 100% (133/133), 7.70 MiB | 22.22 MiB/s, done.
Resolving deltas: 100% (26/26), done.
Error downloading object: data/processed/facility_risk_profiles.csv (385473a): Smudge error: Error downloading data/processed/facility_risk_profiles.csv (385473a51686b29f5d70b5aa56a872d1347274af8e0beac2153ff1cfaf227919): batch response: This repository exceeded its LFS budget. The account responsible for the budget should increase it to restore access.

Errors logged to /content/Team-Q/.git/lfs/logs/20251012T081508.674162597.log
Use `git lfs logs last` to view the log.
error: external filter 'git-lfs filter-process' failed
fatal: data/processed/facility_risk_profiles.csv: smudge filter lfs failed
You can inspect what was checked out with 'git status

In [6]:
%cd Team-Q/disease_trends

/content/Team-Q/disease_trends


In [7]:
%ls

disease_news/  disease_news_classified/  keys.json  LLM-Disease-Trends.ipynb


In [ ]:
import requests
import pandas as pd
import re
import os
import pathlib
from bs4 import BeautifulSoup
import json
from google import genai
from google.genai import types

In [ ]:
def get_full_article(url):
    """
    Scrapes and returns the full text of a news article from the given URL.

    Parameters:
        url (str): URL of the news article.

    Returns:
        str: Concatenated text from all <p> tags if successful.
        None: If the request fails or content is unavailable.
    """
    try:
        response = requests.get(url, timeout=5)
        if response.status_code == 200:
            soup = BeautifulSoup(response.text, 'html.parser')
            paragraphs = soup.find_all('p')
            full_text = "\n".join([p.get_text() for p in paragraphs])
            return full_text
    except requests.exceptions.RequestException:
        return None
    return None


def fetch_news_data(api_key, exec_date, data_path):
    """
    Connects to the news API, retrieves news articles for a given day,
    processes the articles, and saves them as a CSV file.

    Parameters:
        api_key (str): News API key
        exec_date (str): Execution date in YYYY-MM-DD format
        data_path (str): Path to save the data

    Returns:
        str: Path to the saved CSV file
    """
    exec_month = exec_date[:7]  # Extract YYYY-MM
    file_name = f'{exec_date}_news_file.txt'

    # Drop file if it exists
    #if os.path.exists(f"{data_path}{exec_month}/{file_name}"):
        #os.remove(f"{data_path}{exec_month}/{file_name}")

    # Create required directory for the month
    pathlib.Path(f"{data_path}/{exec_month}").mkdir(parents=True, exist_ok=True)

    # Build the news API URL
    url = f"https://newsapi.org/v2/everything?from={exec_date}&to={exec_date}&domains=punchng.com,thenationonlineng.net,vanguardngr.com&language=en&q=(disease OR medicine)&apiKey={api_key}"
    response = requests.get(url)
    resp_dict = response.json()
    articles = resp_dict['articles']

    empty_json = {'date': [], 'content': []}

    # Process each article in the response
    for article in articles:
        published_date = article['publishedAt']
        published_date = re.findall('[\d-]+', published_date)[0]
        content_url = article['url']
        content = get_full_article(content_url)

        # If full content is unavailable, use the description
        if content is None:
            content_descrp = article['description']
            empty_json['content'].append(content_descrp)
        else:
            empty_json['content'].append(content)
        empty_json['date'].append(published_date)

    news_df = pd.DataFrame(empty_json)
    news_df_shape = news_df.shape

    if news_df_shape[0] == 0:
        new_row = [exec_date, "No data for this day"]
        news_df.loc[len(news_df)] = new_row

    output_path = f"{data_path}{exec_month}/{file_name}"
    news_df.to_csv(output_path, index=False)

    return output_path

In [ ]:
with open('keys.json', 'r') as api_keys:
    api_keys=json.load(api_keys)
    news_api_key=api_keys['news_api']
    gemini_api_key=api_keys['gemini']

fetch_news_data(news_api_key, '2025-10-09', 'disease_news')

'disease_news2025-10/2025-10-09_news_file.txt'

In [ ]:
news_df = pd.read_csv('disease_news/2025-10/2025-10-09_news_file.txt')

In [ ]:
news_df

,Unnamed: 0,date,content
0,0,2025-10-09,\n\t\t\t\t\t\n\t\t\t\t\t\t\n\t\t\t\t\t\tFollow...
1,1,2025-10-09,\n reading time 3 minutes\n ...
2,2,2025-10-09,\nAbout us | Advertise with us | Contact us\n ...
3,3,2025-10-09,Research Highlights:\nAdding the novel medicat...
4,4,2025-10-09,Our wellness advice is expert-vetted. Our top ...
...,...,...,...
86,86,2025-10-09,"We're constantly told to ""eat healthy"" – but w..."
87,87,2025-10-09,Mostly law professors | Sometimes contrarian |...
88,88,2025-10-09,Scientists at Washington University School of ...
89,89,2025-10-09,Thank you for visiting nature.com. You are usi...


In [ ]:
def generate_ai_advice(gemini_api_key, news, ai_content_path, exec_date):
    """
    Generates AI-based stock sentiment advice using Google Gemini.

    Parameters:
        gemini_api_key (str): Google Gemini API key
        news (list): List of news content
        ai_content_path (str): Path to save AI content
        exec_date (str): Execution date in YYYY-MM-DD format

    Returns:
        DataFrame with Classified disease
    """
    exec_month = exec_date[:7]
    llm_output = f"{ai_content_path}/{exec_month}/{exec_date}_llm_classified.txt"

    # Create directory if it doesn't exist
    os.makedirs(f"{ai_content_path}/{exec_month}", exist_ok=True)

    # System instruction for the language model
    sys_instruct = "You are an expert disease control agent. For every news, I need you to return ONLY ONE word NOTHING MORE THAN THAT describing what disease is being talked about in the news."
    client = genai.Client(api_key=gemini_api_key)

    diseases = []
    # Format news as text
    #news_as_text = ''
    for ind, cont in enumerate(news.content):
        #news_as_text += f"News_{ind+1}\n{cont}\n\n"

        # Generate content
        response = client.models.generate_content(
            model="gemini-2.0-flash",
            config=types.GenerateContentConfig(
            system_instruction=sys_instruct),
            contents=["""This is the recent news for today. Please follow the instruction below:
                        1. For the news, get me ONLY ONE WORD stating what disease is being talked about
                        2. If it's not a news talking about a disease, return "N/A"
                        3. Strictly follow the instructions here
                        """ + cont]
                        )
        diseases.append(response.text.replace('\n', ''))

    # Save the AI-generated advice to file
    news['disease'] = diseases
    news.to_csv(llm_output, index=False)

    return news

In [ ]:
generate_ai_advice(gemini_api_key, news_df.iloc[:10,1:], 'disease_news_classified', '2025-10-09')

,date,content,disease
0,2025-10-09,\n\t\t\t\t\t\n\t\t\t\t\t\t\n\t\t\t\t\t\tFollow...,Alzheimer's
1,2025-10-09,\n reading time 3 minutes\n ...,Salmonella
2,2025-10-09,\nAbout us | Advertise with us | Contact us\n ...,Alzheimer's
3,2025-10-09,Research Highlights:\nAdding the novel medicat...,Hypertension
4,2025-10-09,Our wellness advice is expert-vetted. Our top ...,N/A
5,2025-10-09,\n reading time 3 minutes\n ...,N/A
6,2025-10-09,"October 9, 2025\n5 min read\nScientists Turned...",Diabetes
7,2025-10-09,Thank you for visiting nature.com. You are usi...,Opioids
8,2025-10-09,\n reading time 3 minutes\n ...,Autism
9,2025-10-09,Advertisement\nThe skeleton of King Richard II...,Periodontal


In [ ]:
Classified = pd.read_csv('disease_news_classified/2025-10/2025-10-09_llm_classified.txt')

In [ ]:
Classified

,date,content,disease
0,2025-10-09,\n\t\t\t\t\t\n\t\t\t\t\t\t\n\t\t\t\t\t\tFollow...,Alzheimer's
1,2025-10-09,\n reading time 3 minutes\n ...,Salmonella
2,2025-10-09,\nAbout us | Advertise with us | Contact us\n ...,Alzheimer's
3,2025-10-09,Research Highlights:\nAdding the novel medicat...,Hypertension
4,2025-10-09,Our wellness advice is expert-vetted. Our top ...,NaN
5,2025-10-09,\n reading time 3 minutes\n ...,NaN
6,2025-10-09,"October 9, 2025\n5 min read\nScientists Turned...",Diabetes
7,2025-10-09,Thank you for visiting nature.com. You are usi...,Opioids
8,2025-10-09,\n reading time 3 minutes\n ...,Autism
9,2025-10-09,Advertisement\nThe skeleton of King Richard II...,Periodontal


In [1]:
%ls

sample_data/


In [2]:
%pwd

'/content'